# 03 — Plots y tablas mínimas para entregar

Objetivo: generar las figuras y tablas básicas:

- evolución de \(T_9(t)\) y \(\rho(t)\);
- evolución de abundancias \(X_i(t)\);
- tabla de abundancias en \(50\), \(200\), \(1000\) s;
- ratios D/H, \(^3\mathrm{He}/\mathrm{H}\), \(^7\mathrm{Li}/\mathrm{H}\), \(^7\mathrm{Be}/\mathrm{H}\);
- checklist para redactar la memoria.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

evolution = pd.read_csv("bbn_evolution.csv")
snapshot_results = pd.read_csv("bbn_snapshot_results.csv")

snapshot_results

In [ ]:
# Figura 1: historia termodinámica impuesta.
fig, ax1 = plt.subplots(figsize=(6.5, 4.2))

ax1.set_xscale("log")
ax1.set_yscale("log")
ax1.plot(evolution["t_s"], evolution["T9"], label=r"$T_9$")
ax1.set_xlabel(r"$t$ [s]")
ax1.set_ylabel(r"$T_9$")

ax2 = ax1.twinx()
ax2.set_yscale("log")
ax2.plot(evolution["t_s"], evolution["rho_g_cm3"], linestyle="--", label=r"$\rho$")
ax2.set_ylabel(r"$\rho$ [g cm$^{-3}$]")

ax1.scatter(snapshot_results["t_s"], snapshot_results["T9"], marker="o")
ax2.scatter(snapshot_results["t_s"], snapshot_results["rho_g_cm3"], marker="s")

ax1.grid(True, which="both", alpha=0.3)
fig.tight_layout()
fig.savefig("fig_thermo_history.pdf", bbox_inches="tight")
plt.show()

print("Guardado: fig_thermo_history.pdf")

In [ ]:
# Figura 2: evolución de abundancias en masa.
# Seleccionamos columnas X_* y quitamos trazas exactamente nulas.
xcols = [c for c in evolution.columns if c.startswith("X_")]
keep = []
for c in xcols:
    if evolution[c].max() > 1e-30:
        keep.append(c)

fig, ax = plt.subplots(figsize=(7.0, 4.8))
ax.set_xscale("log")
ax.set_yscale("log")

for c in keep:
    ax.plot(evolution["t_s"], np.clip(evolution[c], 1e-99, None), label=c.replace("X_", ""))

ax.set_xlabel(r"$t$ [s]")
ax.set_ylabel(r"mass fraction $X_i$")
ax.set_ylim(1e-20, 2)
ax.grid(True, which="both", alpha=0.3)
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
fig.savefig("fig_abundances_evolution.pdf", bbox_inches="tight")
plt.show()

print("Guardado: fig_abundances_evolution.pdf")

In [ ]:
# Figura 3: ratios principales.
ratio_cols = ["D/H", "He3/H", "Li7/H", "Be7/H", "Li7_plus_Be7_over_H"]
ratio_cols = [c for c in ratio_cols if c in snapshot_results.columns]

fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.set_xscale("log")
ax.set_yscale("log")

for c in ratio_cols:
    values = snapshot_results[c].to_numpy(dtype=float)
    if np.any(np.isfinite(values) & (values > 0)):
        ax.plot(snapshot_results["t_s"], values, marker="o", label=c)

ax.set_xlabel(r"$t$ [s]")
ax.set_ylabel("ratio to H")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig("fig_light_element_ratios.pdf", bbox_inches="tight")
plt.show()

print("Guardado: fig_light_element_ratios.pdf")

In [ ]:
# Tabla limpia para la memoria.
cols = [
    "t_s", "T9", "rho_g_cm3", "baryon_sum", "Y_n/Y_p",
    "D/H", "He3/H", "X_He4", "Li7/H", "Be7/H", "Li7_plus_Be7_over_H"
]
cols = [c for c in cols if c in snapshot_results.columns]

table = snapshot_results[cols].copy()

# Formato científico razonable
display(table)

table.to_latex("bbn_snapshot_table.tex", index=False, escape=False, float_format="%.6e")
table.to_csv("bbn_snapshot_table_clean.csv", index=False)

print("Guardado: bbn_snapshot_table.tex")
print("Guardado: bbn_snapshot_table_clean.csv")

In [ ]:
# Checklist mínimo para escribir la memoria.
checklist = {
    "condiciones_termodinamicas_definidas": True,
    "n/p_inicial_1_7": np.isclose(snapshot_results.loc[0, "Y_n/Y_p"], 1/7, rtol=1e-2),
    "conservacion_barionica_aprox": np.allclose(snapshot_results["baryon_sum"], 1.0, rtol=1e-5, atol=1e-8),
    "tabla_snapshots_generada": True,
    "figuras_pdf_generadas": True,
}

for k, v in checklist.items():
    print(f"{k:40s}: {v}")

## Comentario para la memoria

Frase honesta que puedes usar:

> Se ha construido una red nuclear reducida para BBN con `pynucastro`, imponiendo una historia termodinámica \(T(t),\rho(t)\) interpolada a partir de los tres snapshots del enunciado. El cálculo no pretende reemplazar a un código cosmológico completo de BBN, ya que no resuelve de forma autoconsistente la expansión, la termodinámica del plasma ni el desacoplo débil \(n\leftrightarrow p\). Su objetivo es verificar la evolución básica de abundancias ligeras bajo las condiciones prescritas.

Eso te evita vender más de lo que realmente estás haciendo.